# Demo: Vocabulary Middleware (Domain Extended)

This notebook extends [demo-vocabulary-middleware.ipynb](./demo-vocabulary-middleware.ipynb) by adding a small domain-specific biological vocabulary to the indexed vocabulary surface.
The task and middleware pattern stay recognizable; the important change is that the Research Agent can now inspect terms that are actually relevant to the target ontology instead of relying only on generic bundled vocabularies.

---

- **What changed relative to the previous notebook?** `DatasetMiddleware` and `RDFVocabularyMiddleware` are still present, but the vocabulary index now includes a domain-specific TBox alongside the generic bundled vocabularies.
- **What should this condition tell us?** Whether vocabulary access becomes more useful once the available inventory contains terms that actually match the task domain.
- **What is the methodological caveat?** This is a stronger vocabulary condition, not a clean apples-to-apples continuation of the earlier three notebooks. It is best read as a follow-on experiment about vocabulary quality rather than as a directly comparable benchmark row.
- **How should this notebook be read after the first three?** The previous notebook suggested that generic vocabulary access alone was too weakly matched to the biology task to improve grounding. This notebook tests that hypothesis by supplying a small domain-specific vocabulary and asking whether the Research Agent aligns more of the target graph.

## Where does this notebook fit in the series?

| Condition | Middleware | Notebook |
|-----------|-----------|----------|
| Baseline (prompt-only) | None | [demo-baseline-ontology-extraction.ipynb](./demo-baseline-ontology-extraction.ipynb) |
| Dataset middleware | `DatasetMiddleware` | [demo-dataset-middleware.ipynb](./demo-dataset-middleware.ipynb) |
| Vocabulary middleware | `DatasetMiddleware` + `RDFVocabularyMiddleware` | [demo-vocabulary-middleware.ipynb](./demo-vocabulary-middleware.ipynb) |
| **Domain-Specific TBox** | `DatasetMiddleware` + `RDFVocabularyMiddleware` | **this notebook** |

## 1. Setup

We first filter a LangChain-internal warning that is unrelated to the experiment.

In [2]:
import os
import warnings

warnings.filterwarnings(
    "ignore",
    message="Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.",
    category=UserWarning,
)

### 1.1. Bring your own Model

This notebook uses [LangChain's OpenAI integration](https://docs.langchain.com/oss/python/integrations/chat/openai) for [a self-hosted model provider](https://docs.langchain.com/langsmith/playground-model-providers#openai-compatible-endpoint), but you can use any of [LangChain's supported providers](https://docs.langchain.com/oss/python/integrations/providers/all_providers).
What's important is that Deepagents requires a chat model capable of tool invocations.


_NOTE:_ The checked-in run again follows the Ministral model guidance used in the baseline notebook.
`MinistralPromptSuffixMiddleware` remains enabled so that differences between notebooks are driven by dataset affordances rather than by a silent model-prompt regression.


In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    seed=42,
    temperature=1,
)

## 2. The Demonstration

### 2.1. Prompt Setup

The prompt stays deliberately narrow here, just as it did in the previous vocabulary notebook.
We keep `CORE_PROMPT` and the shared stopping rule, and we again remove the prompt-level `VOCABULARY_TIPS` supplement.

That removal is methodological.
In the earlier notebooks, `VOCABULARY_TIPS` helped reduce prompt asymmetry while no explicit vocabulary tool was available.
In the vocabulary-enabled conditions, vocabulary help should come primarily from the middleware's tool surface and indexed term inventory rather than from extra prompt prose.

What this condition **does** receive:

- The shared RDF modeling guidance from `CORE_PROMPT`.
- Middleware-owned dataset tools and middleware-managed RDF session state.
- Read-only vocabulary inspection tools from `RDFVocabularyMiddleware`.
- A domain-specific biological vocabulary added to the indexed vocabulary set.
- Namespace whitelisting that still constrains where new terms may be written.

What this condition **does not** receive:

- The prompt-level `VOCABULARY_TIPS` text used to reduce asymmetry in the earlier notebooks.
- A guarantee that the Research Agent will fully adopt the supplied domain terms just because they are available.

This distinction matters for interpretation.
If alignment improves here, that improvement is easier to attribute to richer vocabulary availability rather than to hidden prompt tuning.
If alignment still stalls, that would suggest the bottleneck lies somewhere deeper than vocabulary discovery alone.

In [4]:
from typing import Final

from demo_utils import CORE_PROMPT, STOPPING_CRITERIA

# SYSTEM_PROMPT: Final[str] = CORE_PROMPT + ONTOLOGY_BRIDGING_TIPS + STOPPING_CRITERIA
SYSTEM_PROMPT: Final[str] = CORE_PROMPT + STOPPING_CRITERIA

### 2.2. Agent Task Definition

We keep the task itself fixed across the series so that the intervention remains middleware composition rather than a rewritten assignment.
The Research Agent still receives the same short passage and the same request to formalize it as RDF.

What changes is the capability mix.
As in the dataset and vocabulary notebooks, the authoritative artifact is the middleware-owned graph rather than a final-answer Turtle block.
Unlike the previous notebook, the agent can now inspect a vocabulary index that includes domain-relevant biology classes in addition to the generic bundled vocabularies.

That makes this notebook a test of vocabulary quality and domain grounding, not merely of syntax robustness or generic vocabulary access.

In [5]:
from demo_utils import TASK

print(TASK)


Please assist me in representing the subject matter of the following text as an RDF graph.
Please reuse established RDF, RDFS, OWL, or other appropriate standard terms
when they fit the source. Please use <urn:ex:> as the base only for IRIs that
you genuinely need to mint as part of your response. Do not invent helper
relations or organizing abstractions unless they are needed to represent an
explicit claim from the text. Be sure to label any new terms or properties that
you mint so that they are human readable.

```text
John is a person. Modern people are classified as _homo sapiens_. Apparently,
homo sapiens falls under the subtribe of Hominina. Every Hominidae is a
Haplorhini, and now the names don't even sound like they're related. Then
we get to primates, which can be controversial for some people, for some reason.
Nobody argues with the idea that primates are mammals, yet some people take
umbrage with the idea that _homo sapiens_ is an animal. Oh, Hominidae contains
Hominina, to

#### 2.2.1. Evaluation Contract

The target graph remains unchanged from the earlier notebooks except for the adjoined renaming of target classes, so the semantic objective stays stable across the series.
As in the other middleware conditions, evaluation uses the middleware-managed graph directly rather than gating on whether the final chat message happened to contain one perfect Turtle block.

That means the main questions here are:

1. Does the run still leave behind a machine-usable RDF graph?
2. Does the richer domain-specific vocabulary inventory increase overlap with the renamed evaluation target?
3. If the graph improves, does the remaining error shift from basic grounding failures toward more debatable modeling choices?

Best-practice documentation triples remain acceptable extras.
The aim is still to assess graph content and modeling choices rather than to punish the run for being readable.

Methodologically, this notebook is best treated as a richer-vocabulary follow-up rather than as a strict statistical continuation of the previous three conditions.

In [6]:
from demo_utils import expected_graph
from rdflib import Graph

print(expected_graph.serialize(format="turtle"))

@prefix ex: <urn:ex:> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Haplorhini a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Primate .

ex:Hominidae a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Haplorhini .

ex:Hominina a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Hominidae .

ex:HomoSapiens a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Hominina .

ex:Mammal a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Animal .

ex:Primate a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Mammal .

ex:John a ex:Person .

ex:Person rdfs:subClassOf ex:HomoSapiens ;
    owl:equivalentClass ex:HomoSapiens .




#### 2.2.2. Domain-Specific Vocabulary Extension

The condition of this notebook is to provide the agent with a set of terms from a domain-specific vocabulary that it can use in its own modeling.
We derive the domain-specific vocabulary directly from the expected output data used throughout the series.

1. The first step in doing this is to actually derive the _new_ target graph, renaming target classes to be from from our domain-specific vocabulary as we do so. This gives us an equivalent output target.
2. Then, we extract those classes to place them into a domain-specific vocabulary. This is an implementation detail that simply prevents us from having to hand-code a domain-specific vocabulary.
3. We DO NOT add any of the optional metadata found in RDF Vocabularies - The agent does no get additional information about the classes beyond what's given in the source text (e.g., lexical surface forms, descriptions, etc). Such information WOULD be accessible in a true RDF Vocabulary.

In [7]:
from rdflib import DC, OWL, RDF, RDFS, Literal, Namespace, URIRef

domain_node = URIRef("https://example.com/ontology")
domain_ns = Namespace(f"{domain_node}#")

pseudo_expected = Graph()
pseudo_expected.bind("exterms", domain_ns)

for t in expected_graph:
    pseudo_expected.add(t)
for n in pseudo_expected.subjects(RDF.type, RDFS.Class):
    assert isinstance(n, URIRef)
    rdfsClass = URIRef(domain_ns[n.removeprefix("urn:ex:")])
    for p, o in pseudo_expected.predicate_objects(n):
        pseudo_expected.remove((n, p, o))
        pseudo_expected.add((rdfsClass, p, o))
    for s, p in pseudo_expected.subject_predicates(n):
        pseudo_expected.remove((s, p, n))
        pseudo_expected.add((s, p, rdfsClass))

print("Expected output graph")
print("-" * 80)
print(pseudo_expected.serialize(format="turtle"))

domain_terms = Graph(identifier=domain_node)
domain_terms.add((domain_node, RDF.type, OWL.Ontology))
domain_terms.add((domain_node, DC.title, Literal("Human Biological Ontology")))
domain_terms.add(
    (domain_node, DC.description, Literal("Biological classes for human ontologies"))
)
domain_terms.bind("exterms", domain_ns)

for rdfsClass in pseudo_expected.subjects(RDF.type, RDFS.Class):
    domain_terms.add((rdfsClass, RDF.type, RDFS.Class))
    domain_terms.add((rdfsClass, RDFS.isDefinedBy, domain_node))

print("Domain terminology supplied to the agent")
print("-" * 80)
print(domain_terms.serialize(format="turtle"))

Expected output graph
--------------------------------------------------------------------------------
@prefix exterms: <https://example.com/ontology#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

exterms:Haplorhini a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf exterms:Primate .

exterms:Hominidae a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf exterms:Haplorhini .

exterms:Hominina a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf exterms:Hominidae .

exterms:HomoSapiens a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf exterms:Hominina .

exterms:Mammal a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf <urn:ex:Animal> .

exterms:Primate a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf exterms:Mammal .

<urn:ex:John> a <urn:ex:Person> .

<urn:ex:Person> rdfs:subClassOf exterms:HomoSapiens ;
    owl:equivalentClass exterms:HomoSapiens .


Domain terminology supplied to the agent
---------------

### 2.3. Example Run

This condition is especially useful to trace because it lets us see whether the Research Agent merely notices the added domain vocabulary or actually uses it while constructing the graph.
`LiveNotebookTrace` remains valuable for the same reason as in the previous notebook: it lets us inspect the agent's decisions rather than only the final dataset.

In this saved run, the added vocabulary is both noticed and used.
The trace opens with a `list_vocabularies` call, which now returns six vocabularies including `Human Biological Ontology` alongside the generic bundled set (RDF, RDFS, OWL, PROV-O, FOAF).
The agent then issues a `list_terms` call scoped to `<https://example.com/ontology>`, walking the six classes of the supplied TBox (`Haplorhini`, `Hominidae`, `Hominina`, `HomoSapiens`, `Mammal`, `Primate`), followed by an `inspect_term` call on `<https://example.com/ontology#HomoSapiens>`.
After that ontology familiarization pass, the agent proceeds to `add_triples` and `serialize_dataset`.
A single whitelist violation is recorded against a typo URI (`http://www.w3.org/1000/01/rdf-schema#subClassOf`) that the agent then corrects.

The resulting behavior is therefore a clear shift from the generic-vocabulary notebook.
The biological taxonomy in the saved graph is anchored in the supplied ontology using `exterms:Haplorhini`, `exterms:Hominidae`, `exterms:Hominina`, `exterms:HomoSapiens`, `exterms:Mammal`, and `exterms:Primate`.
The only local class is `urn:ex:Animal`, which the agent adds because "Animal" is not in the supplied TBox, and John is typed with `foaf:Person` from the bundled FOAF vocabulary rather than with a local `ex:Person`.

In [8]:
from rdflib_reasoning.middleware import (
    VocabularyConfiguration,
    VocabularyDeclaration,
)
from rdflib_reasoning.middleware.namespaces.spec_cache import UserSpec

vocabulary_context = VocabularyConfiguration.bundled_plus(
    VocabularyDeclaration(
        prefix="exterms",
        namespace=domain_node,
        user_spec=UserSpec.from_graph(domain_terms),
    ),
    VocabularyDeclaration(prefix="ex", namespace="urn:ex:"),
).build_context()

In [9]:
from rdflib_reasoning.middleware import (
    RDFVocabularyMiddleware,
    RDFVocabularyMiddlewareConfig,
)
from rdflib_reasoning.middleware.dataset_middleware import (
    DatasetMiddleware,
    DatasetMiddlewareConfig,
)

dataset_middleware = DatasetMiddleware(
    config=DatasetMiddlewareConfig(vocabulary_context=vocabulary_context)
)

vocabulary_middleware = RDFVocabularyMiddleware(
    config=RDFVocabularyMiddlewareConfig(vocabulary_context=vocabulary_context)
)

/Users/roberthall/Projects/rdflib-reasoning/rdflib-reasoning-middleware/src/rdflib_reasoning/middleware/namespaces/spec_normalizer.py:205: NoDefinitionWarning: No valid definition for https://example.com/ontology#Primate found
  definition=_get_term_definition(node, graph).value,
/Users/roberthall/Projects/rdflib-reasoning/rdflib-reasoning-middleware/src/rdflib_reasoning/middleware/namespaces/spec_normalizer.py:205: NoDefinitionWarning: No valid definition for https://example.com/ontology#Haplorhini found
  definition=_get_term_definition(node, graph).value,
/Users/roberthall/Projects/rdflib-reasoning/rdflib-reasoning-middleware/src/rdflib_reasoning/middleware/namespaces/spec_normalizer.py:205: NoDefinitionWarning: No valid definition for https://example.com/ontology#Mammal found
  definition=_get_term_definition(node, graph).value,
/Users/roberthall/Projects/rdflib-reasoning/rdflib-reasoning-middleware/src/rdflib_reasoning/middleware/namespaces/spec_normalizer.py:205: NoDefinitionWarn

In [10]:
display(vocabulary_middleware.list_vocabularies())
display(vocabulary_middleware.list_terms(vocabulary=domain_node))

(VocabularySummary(namespace=rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#'), label='RDF', description='Core RDF data model terms for statements, lists, containers, and literal/datatype machinery.', term_count=31),
 VocabularySummary(namespace=rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#'), label='RDFS', description='Schema-level RDF terms for classes, properties, labels, comments, domain/range, and hierarchy modeling.', term_count=15),
 VocabularySummary(namespace=rdflib.term.URIRef('http://www.w3.org/2002/07/owl#'), label='OWL', description='Ontology modeling and logical constraint terms for classes, restrictions, axioms, and richer property semantics.', term_count=86),
 VocabularySummary(namespace=rdflib.term.URIRef('http://www.w3.org/ns/prov#'), label='PROV-O', description='Provenance terms for entities, activities, agents, and qualified influence relationships.', term_count=94),
 VocabularySummary(namespace=rdflib.term.URIRef('http://xmlns.com/foaf/0

(VocabularyTerm(uri=rdflib.term.URIRef('https://example.com/ontology#Haplorhini'), label='Haplorhini', definition='<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Haplorhini</literal_definition_missing>', termType=<VocabularyTermType.CLASS: 'class'>),
 VocabularyTerm(uri=rdflib.term.URIRef('https://example.com/ontology#Hominidae'), label='Hominidae', definition='<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Hominidae</literal_definition_missing>', termType=<VocabularyTermType.CLASS: 'class'>),
 VocabularyTerm(uri=rdflib.term.URIRef('https://example.com/ontology#Hominina'), label='Hominina', definition='<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Hominina</literal_definition_missing>', termType=<VocabularyTermType.CLASS: 'class'>),
 VocabularyTerm(uri=rdflib.term.URIRef('https://example.com/ontology#HomoSapiens'), label='HomoSapiens', definition

In [11]:
import logging
import time
import uuid

from deepagents import create_deep_agent
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.errors import GraphRecursionError
from openai import APIConnectionError, APITimeoutError, BadRequestError, RateLimitError
from rdflib import Namespace
from rdflib_reasoning.middleware.continuation_guard_middleware import (
    ContinuationGuardMiddleware,
)
from rdflib_reasoning.middleware.ministral_middleware import (
    MinistralPromptSuffixMiddleware,
)
from rdflib_reasoning.middleware.tracing_notebook import LiveNotebookTrace

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("rdflib_reasoning.middleware").setLevel(logging.DEBUG)
logger = logging.getLogger(__name__)


# NOTE: other ontologies include BFO or NCIT

with LiveNotebookTrace(heading="Agent Activity") as trace:
    try:
        agent = trace.attach(
            create_deep_agent(
                model=llm,
                system_prompt=SYSTEM_PROMPT,
                middleware=[
                    dataset_middleware,
                    vocabulary_middleware,
                    MinistralPromptSuffixMiddleware(),
                    ContinuationGuardMiddleware(),
                ],
                checkpointer=MemorySaver(),
            )
        )

        thread_id = str(uuid.uuid4())
        run_config = {"configurable": {"thread_id": thread_id}}
        run_input = {"messages": [HumanMessage(content=TASK)]}

        attempt = -1
        for attempt in range(3):
            try:
                agent_response = agent.invoke(run_input, config=run_config)
                break
            except GraphRecursionError:
                display(trace.sink.snapshot())
                raise
            except (RateLimitError, APIConnectionError, APITimeoutError) as e:
                logger.info("Transient error: %s", e)
                time.sleep(2**attempt)
                # run_input = None  # None -> resume same run; fails if previous was AI assistant
                run_input["messages"] = [
                    HumanMessage(
                        content=(
                            "[rdflib_reasoning-recovery] "
                            "Continue from where you left off. Do not repeat completed tool calls. "
                            "If the dataset is already complete, return the final Turtle answer now. "
                            "Otherwise emit the next needed tool call."
                        )
                    )
                ]
            except BadRequestError:
                display(trace.sink.snapshot())
                raise
        else:
            raise RuntimeError(f"Agent failed after {attempt + 1} retries")
    finally:
        trace.dump("demo-vocabulary-middleware-ext.trace.json")

# Agent Activity

## Turn 1

### Model Input Summary

- `system`: You are a research assistant specialized in knowledge graphs and the Semantic Web. Task objective: - Represent the provided source text as RDF, grounded in explicit claims from that text. - Do not introduce uncertain or speculative facts...
- `human`: Please assist me in representing the subject matter of the following text as an RDF graph. Please reuse established RDF, RDFS, OWL, or other appropriate standard terms when they fit the source. Please use <urn:ex:> as the base only for I...

### Tool: list_vocabularies

#### Arguments

```json
{}
```

#### Result

```json
{"vocabularies":[{"namespace":"<http://www.w3.org/1999/02/22-rdf-syntax-ns#>","label":"RDF","description":"Core RDF data model terms for statements, lists, containers, and literal/datatype machinery.","term_count":31},{"namespace":"<http://www.w3.org/2000/01/rdf-schema#>","label":"RDFS","description":"Schema-level RDF terms for classes, properties, labels, comments, domain/range, and hierarchy modeling.","term_count":15},{"namespace":"<http://www.w3.org/2002/07/owl#>","label":"OWL","description":"Ontology modeling and logical constraint terms for classes, restrictions, axioms, and richer property semantics.","term_count":86},{"namespace":"<http://www.w3.org/ns/prov#>","label":"PROV-O","description":"Provenance terms for entities, activities, agents, and qualified influence relationships.","term_count":94},{"namespace":"<http://xmlns.com/foaf/0.1/>","label":"FOAF","description":"People, agents, profiles, social connections, and related online identity and metadata terms.","term_count":72},{"namespace":"<https://example.com/ontology>","label":"Human Biological Ontology","description":"Biological classes for human ontologies","term_count":6}]}
```

#### Message

```json
{
  "content": "{\"vocabularies\":[{\"namespace\":\"<http://www.w3.org/1999/02/22-rdf-syntax-ns#>\",\"label\":\"RDF\",\"description\":\"Core RDF data model terms for statements, lists, containers, and literal/datatype machinery.\",\"term_count\":31},{\"namespace\":\"<http://www.w3.org/2000/01/rdf-schema#>\",\"label\":\"RDFS\",\"description\":\"Schema-level RDF terms for classes, properties, labels, comments, domain/range, and hierarchy modeling.\",\"term_count\":15},{\"namespace\":\"<http://www.w3.org/2002/07/owl#>\",\"label\":\"OWL\",\"description\":\"Ontology modeling and logical constraint terms for classes, restrictions, axioms, and richer property semantics.\",\"term_count\":86},{\"namespace\":\"<http://www.w3.org/ns/prov#>\",\"label\":\"PROV-O\",\"description\":\"Provenance terms for entities, activities, agents, and qualified influence relationships.\",\"term_count\":94},{\"namespace\":\"<http://xmlns.com/foaf/0.1/>\",\"label\":\"FOAF\",\"description\":\"People, agents, profiles, social connections, and related online identity and metadata terms.\",\"term_count\":72},{\"namespace\":\"<https://example.com/ontology>\",\"label\":\"Human Biological Ontology\",\"description\":\"Biological classes for human ontologies\",\"term_count\":6}]}",
  "status": "success",
  "tool_call_id": "zIiULxodt"
}
```

## Turn 2

### Model Input Summary

- `system`: You are a research assistant specialized in knowledge graphs and the Semantic Web. Task objective: - Represent the provided source text as RDF, grounded in explicit claims from that text. - Do not introduce uncertain or speculative facts...
- `human`: Please assist me in representing the subject matter of the following text as an RDF graph. Please reuse established RDF, RDFS, OWL, or other appropriate standard terms when they fit the source. Please use <urn:ex:> as the base only for I...
- `ai`: Let me analyze the source text to identify explicit claims and model them faithfully in RDF. I need to determine which concepts can be represented using existing vocabularies and only mint local IRIs when genuinely required. Step 1: Iden...
- `tool`: {"vocabularies":[{"namespace":"<http://www.w3.org/1999/02/22-rdf-syntax-ns#>","label":"RDF","description":"Core RDF data model terms for statements, lists, containers, and literal/datatype machinery.","term_count":31},{"namespace":"<http...

### Tool: list_terms

#### Arguments

```json
{
  "limit": 20,
  "term_type": "class",
  "vocabulary": "<https://example.com/ontology>"
}
```

#### Result

```json
[{"definition": "<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Haplorhini</literal_definition_missing>", "label": "Haplorhini", "termType": "class", "uri": "<https://example.com/ontology#Haplorhini>"}, {"definition": "<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Hominidae</literal_definition_missing>", "label": "Hominidae", "termType": "class", "uri": "<https://example.com/ontology#Hominidae>"}, {"definition": "<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Hominina</literal_definition_missing>", "label": "Hominina", "termType": "class", "uri": "<https://example.com/ontology#Hominina>"}, {"definition": "<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#HomoSapiens</literal_definition_missing>", "label": "HomoSapiens", "termType": "class", "uri": "<https://example.com/ontology#HomoSapiens>"}, {"definition": "<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Mammal</literal_definition_missing>", "label": "Mammal", "termType": "class", "uri": "<https://example.com/ontology#Mammal>"}, {"definition": "<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Primate</literal_definition_missing>", "label": "Primate", "termType": "class", "uri": "<https://example.com/ontology#Primate>"}]
```

#### Message

```json
{
  "content": "[{\"definition\": \"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Haplorhini</literal_definition_missing>\", \"label\": \"Haplorhini\", \"termType\": \"class\", \"uri\": \"<https://example.com/ontology#Haplorhini>\"}, {\"definition\": \"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Hominidae</literal_definition_missing>\", \"label\": \"Hominidae\", \"termType\": \"class\", \"uri\": \"<https://example.com/ontology#Hominidae>\"}, {\"definition\": \"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Hominina</literal_definition_missing>\", \"label\": \"Hominina\", \"termType\": \"class\", \"uri\": \"<https://example.com/ontology#Hominina>\"}, {\"definition\": \"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#HomoSapiens</literal_definition_missing>\", \"label\": \"HomoSapiens\", \"termType\": \"class\", \"uri\": \"<https://example.com/ontology#HomoSapiens>\"}, {\"definition\": \"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Mammal</literal_definition_missing>\", \"label\": \"Mammal\", \"termType\": \"class\", \"uri\": \"<https://example.com/ontology#Mammal>\"}, {\"definition\": \"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Primate</literal_definition_missing>\", \"label\": \"Primate\", \"termType\": \"class\", \"uri\": \"<https://example.com/ontology#Primate>\"}]",
  "status": "success",
  "tool_call_id": "niGT0wKwF"
}
```

## Turn 3

### Model Input Summary

- `system`: You are a research assistant specialized in knowledge graphs and the Semantic Web. Task objective: - Represent the provided source text as RDF, grounded in explicit claims from that text. - Do not introduce uncertain or speculative facts...
- `human`: Please assist me in representing the subject matter of the following text as an RDF graph. Please reuse established RDF, RDFS, OWL, or other appropriate standard terms when they fit the source. Please use <urn:ex:> as the base only for I...
- `ai`: Let me analyze the source text to identify explicit claims and model them faithfully in RDF. I need to determine which concepts can be represented using existing vocabularies and only mint local IRIs when genuinely required. Step 1: Iden...
- `tool`: {"vocabularies":[{"namespace":"<http://www.w3.org/1999/02/22-rdf-syntax-ns#>","label":"RDF","description":"Core RDF data model terms for statements, lists, containers, and literal/datatype machinery.","term_count":31},{"namespace":"<http...
- `ai`: I see that there is an indexed vocabulary called "Human Biological Ontology" with 6 terms. This may contain some of the taxonomic classes (e.g., "homo sapiens", "Hominidae", etc.). I should inspect its class terms to see if it covers the...
- `tool`: [{"definition": "<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Haplorhini</literal_definition_missing>", "label": "Haplorhini", "termType": "class", "uri": "<https://example.com/ontology#Ha...

### Tool: inspect_term

#### Arguments

```json
{
  "include_source_rdf": true,
  "term": "<https://example.com/ontology#HomoSapiens>"
}
```

#### Result

```json
{"uri":"<https://example.com/ontology#HomoSapiens>","label":"HomoSapiens","definition":"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#HomoSapiens</literal_definition_missing>","termType":"class","vocabulary":"<https://example.com/ontology>","superTerms":[],"domain":[],"range":[],"source_rdf":{"format":"turtle","content":"@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n\n<https://example.com/ontology#HomoSapiens> a rdfs:Class ;\n    rdfs:isDefinedBy <https://example.com/ontology> .\n\n","default_graph_triple_count":2,"is_empty":false,"message":"Serialized the current default graph containing 2 triples."}}
```

#### Message

```json
{
  "content": "{\"uri\":\"<https://example.com/ontology#HomoSapiens>\",\"label\":\"HomoSapiens\",\"definition\":\"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#HomoSapiens</literal_definition_missing>\",\"termType\":\"class\",\"vocabulary\":\"<https://example.com/ontology>\",\"superTerms\":[],\"domain\":[],\"range\":[],\"source_rdf\":{\"format\":\"turtle\",\"content\":\"@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\\n\\n<https://example.com/ontology#HomoSapiens> a rdfs:Class ;\\n    rdfs:isDefinedBy <https://example.com/ontology> .\\n\\n\",\"default_graph_triple_count\":2,\"is_empty\":false,\"message\":\"Serialized the current default graph containing 2 triples.\"}}",
  "status": "success",
  "tool_call_id": "G5UwEOY8E"
}
```

## Turn 4

### Model Input Summary

- `system`: You are a research assistant specialized in knowledge graphs and the Semantic Web. Task objective: - Represent the provided source text as RDF, grounded in explicit claims from that text. - Do not introduce uncertain or speculative facts...
- `human`: Please assist me in representing the subject matter of the following text as an RDF graph. Please reuse established RDF, RDFS, OWL, or other appropriate standard terms when they fit the source. Please use <urn:ex:> as the base only for I...
- `ai`: Let me analyze the source text to identify explicit claims and model them faithfully in RDF. I need to determine which concepts can be represented using existing vocabularies and only mint local IRIs when genuinely required. Step 1: Iden...
- `tool`: {"vocabularies":[{"namespace":"<http://www.w3.org/1999/02/22-rdf-syntax-ns#>","label":"RDF","description":"Core RDF data model terms for statements, lists, containers, and literal/datatype machinery.","term_count":31},{"namespace":"<http...
- `ai`: I see that there is an indexed vocabulary called "Human Biological Ontology" with 6 terms. This may contain some of the taxonomic classes (e.g., "homo sapiens", "Hominidae", etc.). I should inspect its class terms to see if it covers the...
- `tool`: [{"definition": "<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Haplorhini</literal_definition_missing>", "label": "Haplorhini", "termType": "class", "uri": "<https://example.com/ontology#Ha...
- `ai`: I see that the Human Biological Ontology includes the classes Haplorhini, Hominidae, Hominina, HomoSapiens (which matches the text's "homo sapiens"), Mammal, and Primate. These match the main taxonomic terms in the source text, except fo...
- `tool`: {"uri":"<https://example.com/ontology#HomoSapiens>","label":"HomoSapiens","definition":"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#HomoSapiens</literal_definition_missing>","termType":"c...

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "\"Animal\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Animal"
    },
    {
      "object": "\"A living organism capable of loci, typically multi c\u00e9llulare.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Animal"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Animal"
    }
  ]
}
```

#### Result

```json
requested=3 updated=3 unchanged=0 no_action_needed=False message='3 of 3 triples added.'
```

#### Message

```json
{
  "content": "requested=3 updated=3 unchanged=0 no_action_needed=False message='3 of 3 triples added.'",
  "status": "success",
  "tool_call_id": "ITWvlVCX3"
}
```

## Turn 5

### Model Input Summary

- `system`: You are a research assistant specialized in knowledge graphs and the Semantic Web. Task objective: - Represent the provided source text as RDF, grounded in explicit claims from that text. - Do not introduce uncertain or speculative facts...
- `human`: Please assist me in representing the subject matter of the following text as an RDF graph. Please reuse established RDF, RDFS, OWL, or other appropriate standard terms when they fit the source. Please use <urn:ex:> as the base only for I...
- `ai`: Let me analyze the source text to identify explicit claims and model them faithfully in RDF. I need to determine which concepts can be represented using existing vocabularies and only mint local IRIs when genuinely required. Step 1: Iden...
- `tool`: {"vocabularies":[{"namespace":"<http://www.w3.org/1999/02/22-rdf-syntax-ns#>","label":"RDF","description":"Core RDF data model terms for statements, lists, containers, and literal/datatype machinery.","term_count":31},{"namespace":"<http...
- `ai`: I see that there is an indexed vocabulary called "Human Biological Ontology" with 6 terms. This may contain some of the taxonomic classes (e.g., "homo sapiens", "Hominidae", etc.). I should inspect its class terms to see if it covers the...
- `tool`: [{"definition": "<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Haplorhini</literal_definition_missing>", "label": "Haplorhini", "termType": "class", "uri": "<https://example.com/ontology#Ha...
- `ai`: I see that the Human Biological Ontology includes the classes Haplorhini, Hominidae, Hominina, HomoSapiens (which matches the text's "homo sapiens"), Mammal, and Primate. These match the main taxonomic terms in the source text, except fo...
- `tool`: {"uri":"<https://example.com/ontology#HomoSapiens>","label":"HomoSapiens","definition":"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#HomoSapiens</literal_definition_missing>","termType":"c...
- `ai`: I see that the indexed term for HomoSapiens does not have a definition or label in the source RDF, only an rdfs:isDefinedBy. To be consistent, I will use the provided label "HomoSapiens" in my RDF, using the indexed Einkhorn. I will use ...
- `tool`: requested=3 updated=3 unchanged=0 no_action_needed=False message='3 of 3 triples added.'

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://xmlns.com/foaf/0.1/Person",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:john"
    },
    {
      "object": "\"John\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:john"
    },
    {
      "object": "<https://example.com/ontology#Hominina>",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#HomoSapiens>"
    },
    {
      "object": "<https://example.com/ontology#Mammal>",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#HomoSapiens>"
    },
    {
      "object": "urn:ex:Animal",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#HomoSapiens>"
    },
    {
      "object": "<https://example.com/ontology#Hominidae>",
      "predicate": "http://www.w3.org/1000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#Hominina>"
    },
    {
      "object": "<https://example.com/ontology#Haplorhini>",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#Hominidae>"
    },
    {
      "object": "<https://example.com/ontology#Primate>",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#Hominidae>"
    },
    {
      "object": "<https://example.com/ontology#Mammal>",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#Primate>"
    },
    {
      "object": "urn:ex:Animal",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#Mammal>"
    }
  ]
}
```

#### Rejection

```json
{
  "content": "The term http://www.w3.org/1000/01/rdf-schema#subClassOf is not in the vocabulary whitelist for this knowledge base. You SHOULD verify that you are using the correct term or consider the alternatives below. You MUST NOT call `add_triples` again with this same IRI.\n\nNo close matches were found. You MUST use a different term from the allowed vocabularies listed in the system prompt.",
  "status": "error",
  "tool_call_id": "yauK69F6g"
}
```

## Turn 6

### Model Input Summary

- `system`: You are a research assistant specialized in knowledge graphs and the Semantic Web. Task objective: - Represent the provided source text as RDF, grounded in explicit claims from that text. - Do not introduce uncertain or speculative facts...
- `human`: Please assist me in representing the subject matter of the following text as an RDF graph. Please reuse established RDF, RDFS, OWL, or other appropriate standard terms when they fit the source. Please use <urn:ex:> as the base only for I...
- `ai`: Let me analyze the source text to identify explicit claims and model them faithfully in RDF. I need to determine which concepts can be represented using existing vocabularies and only mint local IRIs when genuinely required. Step 1: Iden...
- `tool`: {"vocabularies":[{"namespace":"<http://www.w3.org/1999/02/22-rdf-syntax-ns#>","label":"RDF","description":"Core RDF data model terms for statements, lists, containers, and literal/datatype machinery.","term_count":31},{"namespace":"<http...
- `ai`: I see that there is an indexed vocabulary called "Human Biological Ontology" with 6 terms. This may contain some of the taxonomic classes (e.g., "homo sapiens", "Hominidae", etc.). I should inspect its class terms to see if it covers the...
- `tool`: [{"definition": "<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Haplorhini</literal_definition_missing>", "label": "Haplorhini", "termType": "class", "uri": "<https://example.com/ontology#Ha...
- `ai`: I see that the Human Biological Ontology includes the classes Haplorhini, Hominidae, Hominina, HomoSapiens (which matches the text's "homo sapiens"), Mammal, and Primate. These match the main taxonomic terms in the source text, except fo...
- `tool`: {"uri":"<https://example.com/ontology#HomoSapiens>","label":"HomoSapiens","definition":"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#HomoSapiens</literal_definition_missing>","termType":"c...
- `ai`: I see that the indexed term for HomoSapiens does not have a definition or label in the source RDF, only an rdfs:isDefinedBy. To be consistent, I will use the provided label "HomoSapiens" in my RDF, using the indexed Einkhorn. I will use ...
- `tool`: requested=3 updated=3 unchanged=0 no_action_needed=False message='3 of 3 triples added.'
- `ai`: Let me now serialize the dataset to finalize the RDF graph, which will include: - John as a foaf:Person. - HomoSapiens, Hominidae, Hominina, Haplorhini, Mammal, and Primate, all using the indexed Einkhorns. - The class alternates: Mammal...
- `tool`: The term http://www.w3.org/1000/01/rdf-schema#subClassOf is not in the vocabulary whitelist for this knowledge base. You SHOULD verify that you are using the correct term or consider the alternatives below. You MUST NOT call `add_triples...

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://xmlns.com/foaf/0.1/Person",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:john"
    },
    {
      "object": "\"John\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:john"
    },
    {
      "object": "<https://example.com/ontology#Hominina>",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#HomoSapiens>"
    },
    {
      "object": "<https://example.com/ontology#Mammal>",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#HomoSapiens>"
    },
    {
      "object": "urn:ex:Animal",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#HomoSapiens>"
    },
    {
      "object": "<https://example.com/ontology#Hominidae>",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#Hominina>"
    },
    {
      "object": "<https://example.com/ontology#Haplorhini>",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#Hominidae>"
    },
    {
      "object": "<https://example.com/ontology#Primate>",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#Hominidae>"
    },
    {
      "object": "<https://example.com/ontology#Mammal>",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#Primate>"
    },
    {
      "object": "urn:ex:Animal",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "<https://example.com/ontology#Mammal>"
    }
  ]
}
```

#### Result

```json
requested=10 updated=5 unchanged=5 no_action_needed=False message='5 of 10 triples added. 5 of 10 triples were already present.'
```

#### Message

```json
{
  "content": "requested=10 updated=5 unchanged=5 no_action_needed=False message='5 of 10 triples added. 5 of 10 triples were already present.'",
  "status": "success",
  "tool_call_id": "djkjrJMpv"
}
```

## Turn 7

### Model Input Summary

- `system`: You are a research assistant specialized in knowledge graphs and the Semantic Web. Task objective: - Represent the provided source text as RDF, grounded in explicit claims from that text. - Do not introduce uncertain or speculative facts...
- `human`: Please assist me in representing the subject matter of the following text as an RDF graph. Please reuse established RDF, RDFS, OWL, or other appropriate standard terms when they fit the source. Please use <urn:ex:> as the base only for I...
- `ai`: Let me analyze the source text to identify explicit claims and model them faithfully in RDF. I need to determine which concepts can be represented using existing vocabularies and only mint local IRIs when genuinely required. Step 1: Iden...
- `tool`: {"vocabularies":[{"namespace":"<http://www.w3.org/1999/02/22-rdf-syntax-ns#>","label":"RDF","description":"Core RDF data model terms for statements, lists, containers, and literal/datatype machinery.","term_count":31},{"namespace":"<http...
- `ai`: I see that there is an indexed vocabulary called "Human Biological Ontology" with 6 terms. This may contain some of the taxonomic classes (e.g., "homo sapiens", "Hominidae", etc.). I should inspect its class terms to see if it covers the...
- `tool`: [{"definition": "<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Haplorhini</literal_definition_missing>", "label": "Haplorhini", "termType": "class", "uri": "<https://example.com/ontology#Ha...
- `ai`: I see that the Human Biological Ontology includes the classes Haplorhini, Hominidae, Hominina, HomoSapiens (which matches the text's "homo sapiens"), Mammal, and Primate. These match the main taxonomic terms in the source text, except fo...
- `tool`: {"uri":"<https://example.com/ontology#HomoSapiens>","label":"HomoSapiens","definition":"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#HomoSapiens</literal_definition_missing>","termType":"c...
- `ai`: I see that the indexed term for HomoSapiens does not have a definition or label in the source RDF, only an rdfs:isDefinedBy. To be consistent, I will use the provided label "HomoSapiens" in my RDF, using the indexed Einkhorn. I will use ...
- `tool`: requested=3 updated=3 unchanged=0 no_action_needed=False message='3 of 3 triples added.'
- `ai`: Let me now serialize the dataset to finalize the RDF graph, which will include: - John as a foaf:Person. - HomoSapiens, Hominidae, Hominina, Haplorhini, Mammal, and Primate, all using the indexed Einkhorns. - The class alternates: Mammal...
- `tool`: The term http://www.w3.org/1000/01/rdf-schema#subClassOf is not in the vocabulary whitelist for this knowledge base. You SHOULD verify that you are using the correct term or consider the alternatives below. You MUST NOT call `add_triples...
- `ai`: I see that I made an error in the IRI for rdfs:subClassOf. The correct IRI is http://www.w3.org/2000/01/rdf-s schema#subClassOf (note the missing zero in the year). Let me fix that and add the triples again.
- `tool`: requested=10 updated=5 unchanged=5 no_action_needed=False message='5 of 10 triples added. 5 of 10 triples were already present.'

### Tool: serialize_dataset

#### Arguments

```json
{
  "format": "turtle"
}
```

#### Result

```json
format='turtle' content='@prefix foaf: <http://xmlns.com/foaf/0.1/> .\n@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n\n<urn:ex:Animal> a rdfs:Class ;\n    rdfs:label "Animal" ;\n    rdfs:comment "A living organism capable of loci, typically multi céllulare." .\n\n<https://example.com/ontology#HomoSapiens> rdfs:subClassOf <https://example.com/ontology#Hominina>,\n        <https://example.com/ontology#Mammal>,\n        <urn:ex:Animal> .\n\n<urn:ex:john> a foaf:Person ;\n    rdfs:label "John" .\n\n<https://example.com/ontology#Hominidae> rdfs:subClassOf <https://example.com/ontology#Haplorhini>,\n        <https://example.com/ontology#Primate> .\n\n<https://example.com/ontology#Hominina> rdfs:subClassOf <https://example.com/ontology#Hominidae> .\n\n<https://example.com/ontology#Primate> rdfs:subClassOf <https://example.com/ontology#Mammal> .\n\n<https://example.com/ontology#Mammal> rdfs:subClassOf <urn:ex:Animal> .\n\n' default_graph_triple_count=13 is_empty=False message='Serialized the current default graph containing 13 triples.'
```

#### Message

```json
{
  "content": "format='turtle' content='@prefix foaf: <http://xmlns.com/foaf/0.1/> .\\n@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\\n\\n<urn:ex:Animal> a rdfs:Class ;\\n    rdfs:label \"Animal\" ;\\n    rdfs:comment \"A living organism capable of loci, typically multi c\u00e9llulare.\" .\\n\\n<https://example.com/ontology#HomoSapiens> rdfs:subClassOf <https://example.com/ontology#Hominina>,\\n        <https://example.com/ontology#Mammal>,\\n        <urn:ex:Animal> .\\n\\n<urn:ex:john> a foaf:Person ;\\n    rdfs:label \"John\" .\\n\\n<https://example.com/ontology#Hominidae> rdfs:subClassOf <https://example.com/ontology#Haplorhini>,\\n        <https://example.com/ontology#Primate> .\\n\\n<https://example.com/ontology#Hominina> rdfs:subClassOf <https://example.com/ontology#Hominidae> .\\n\\n<https://example.com/ontology#Primate> rdfs:subClassOf <https://example.com/ontology#Mammal> .\\n\\n<https://example.com/ontology#Mammal> rdfs:subClassOf <urn:ex:Animal> .\\n\\n' default_graph_triple_count=13 is_empty=False message='Serialized the current default graph containing 13 triples.'",
  "status": "success",
  "tool_call_id": "H7evBwZQF"
}
```

## Turn 8

### Model Input Summary

- `system`: You are a research assistant specialized in knowledge graphs and the Semantic Web. Task objective: - Represent the provided source text as RDF, grounded in explicit claims from that text. - Do not introduce uncertain or speculative facts...
- `human`: Please assist me in representing the subject matter of the following text as an RDF graph. Please reuse established RDF, RDFS, OWL, or other appropriate standard terms when they fit the source. Please use <urn:ex:> as the base only for I...
- `ai`: Let me analyze the source text to identify explicit claims and model them faithfully in RDF. I need to determine which concepts can be represented using existing vocabularies and only mint local IRIs when genuinely required. Step 1: Iden...
- `tool`: {"vocabularies":[{"namespace":"<http://www.w3.org/1999/02/22-rdf-syntax-ns#>","label":"RDF","description":"Core RDF data model terms for statements, lists, containers, and literal/datatype machinery.","term_count":31},{"namespace":"<http...
- `ai`: I see that there is an indexed vocabulary called "Human Biological Ontology" with 6 terms. This may contain some of the taxonomic classes (e.g., "homo sapiens", "Hominidae", etc.). I should inspect its class terms to see if it covers the...
- `tool`: [{"definition": "<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Haplorhini</literal_definition_missing>", "label": "Haplorhini", "termType": "class", "uri": "<https://example.com/ontology#Ha...
- `ai`: I see that the Human Biological Ontology includes the classes Haplorhini, Hominidae, Hominina, HomoSapiens (which matches the text's "homo sapiens"), Mammal, and Primate. These match the main taxonomic terms in the source text, except fo...
- `tool`: {"uri":"<https://example.com/ontology#HomoSapiens>","label":"HomoSapiens","definition":"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#HomoSapiens</literal_definition_missing>","termType":"c...
- `ai`: I see that the indexed term for HomoSapiens does not have a definition or label in the source RDF, only an rdfs:isDefinedBy. To be consistent, I will use the provided label "HomoSapiens" in my RDF, using the indexed Einkhorn. I will use ...
- `tool`: requested=3 updated=3 unchanged=0 no_action_needed=False message='3 of 3 triples added.'
- `ai`: Let me now serialize the dataset to finalize the RDF graph, which will include: - John as a foaf:Person. - HomoSapiens, Hominidae, Hominina, Haplorhini, Mammal, and Primate, all using the indexed Einkhorns. - The class alternates: Mammal...
- `tool`: The term http://www.w3.org/1000/01/rdf-schema#subClassOf is not in the vocabulary whitelist for this knowledge base. You SHOULD verify that you are using the correct term or consider the alternatives below. You MUST NOT call `add_triples...
- `ai`: I see that I made an error in the IRI for rdfs:subClassOf. The correct IRI is http://www.w3.org/2000/01/rdf-s schema#subClassOf (note the missing zero in the year). Let me fix that and add the triples again.
- `tool`: requested=10 updated=5 unchanged=5 no_action_needed=False message='5 of 10 triples added. 5 of 10 triples were already present.'
- `ai`: Let me now serialize the dataset to finalize the RDF graph, which will include: - John as a foaf:Person. - HomoSapiens, Hominidae, Hominina, Haplorhini, Mammal, and Primate, all using the indexed Einkhorns. - The class alternates: Mammal...
- `tool`: format='turtle' content='@prefix foaf: <http://xmlns.com/foaf/0.1/> .\n@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n\n<urn:ex:Animal> a rdfs:Class ;\n rdfs:label "Animal" ;\n rdfs:comment "A living organism capable of loci, t...

### Tool: serialize_dataset

#### Arguments

```json
{
  "format": "turtle"
}
```

## Turn 9

### Model Input Summary

- `system`: You are a research assistant specialized in knowledge graphs and the Semantic Web. Task objective: - Represent the provided source text as RDF, grounded in explicit claims from that text. - Do not introduce uncertain or speculative facts...
- `human`: Please assist me in representing the subject matter of the following text as an RDF graph. Please reuse established RDF, RDFS, OWL, or other appropriate standard terms when they fit the source. Please use <urn:ex:> as the base only for I...
- `ai`: Let me analyze the source text to identify explicit claims and model them faithfully in RDF. I need to determine which concepts can be represented using existing vocabularies and only mint local IRIs when genuinely required. Step 1: Iden...
- `tool`: {"vocabularies":[{"namespace":"<http://www.w3.org/1999/02/22-rdf-syntax-ns#>","label":"RDF","description":"Core RDF data model terms for statements, lists, containers, and literal/datatype machinery.","term_count":31},{"namespace":"<http...
- `ai`: I see that there is an indexed vocabulary called "Human Biological Ontology" with 6 terms. This may contain some of the taxonomic classes (e.g., "homo sapiens", "Hominidae", etc.). I should inspect its class terms to see if it covers the...
- `tool`: [{"definition": "<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Haplorhini</literal_definition_missing>", "label": "Haplorhini", "termType": "class", "uri": "<https://example.com/ontology#Ha...
- `ai`: I see that the Human Biological Ontology includes the classes Haplorhini, Hominidae, Hominina, HomoSapiens (which matches the text's "homo sapiens"), Mammal, and Primate. These match the main taxonomic terms in the source text, except fo...
- `tool`: {"uri":"<https://example.com/ontology#HomoSapiens>","label":"HomoSapiens","definition":"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#HomoSapiens</literal_definition_missing>","termType":"c...
- `ai`: I see that the indexed term for HomoSapiens does not have a definition or label in the source RDF, only an rdfs:isDefinedBy. To be consistent, I will use the provided label "HomoSapiens" in my RDF, using the indexed Einkhorn. I will use ...
- `tool`: requested=3 updated=3 unchanged=0 no_action_needed=False message='3 of 3 triples added.'
- `ai`: Let me now serialize the dataset to finalize the RDF graph, which will include: - John as a foaf:Person. - HomoSapiens, Hominidae, Hominina, Haplorhini, Mammal, and Primate, all using the indexed Einkhorns. - The class alternates: Mammal...
- `tool`: The term http://www.w3.org/1000/01/rdf-schema#subClassOf is not in the vocabulary whitelist for this knowledge base. You SHOULD verify that you are using the correct term or consider the alternatives below. You MUST NOT call `add_triples...
- `ai`: I see that I made an error in the IRI for rdfs:subClassOf. The correct IRI is http://www.w3.org/2000/01/rdf-s schema#subClassOf (note the missing zero in the year). Let me fix that and add the triples again.
- `tool`: requested=10 updated=5 unchanged=5 no_action_needed=False message='5 of 10 triples added. 5 of 10 triples were already present.'
- `ai`: Let me now serialize the dataset to finalize the RDF graph, which will include: - John as a foaf:Person. - HomoSapiens, Hominidae, Hominina, Haplorhini, Mammal, and Primate, all using the indexed Einkhorns. - The class alternates: Mammal...
- `tool`: format='turtle' content='@prefix foaf: <http://xmlns.com/foaf/0.1/> .\n@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n\n<urn:ex:Animal> a rdfs:Class ;\n rdfs:label "Animal" ;\n rdfs:comment "A living organism capable of loci, t...
- `ai`: I will now serialize the dataset to finalize the RDF graph, which will include: - John as a foaf:Person. - HomoSapiens, Hominidae, Hominina, Haplorhini, Mammal, and Primate, all using the indexed Einkhorns. - The class alternates: Mammal...
- `tool`: The dataset has not changed since the previous `serialize_dataset` call in this format. Re-serializing will not reformat or improve the graph. Use the previous successful serialization as your final answer if it already reflects the grap...

### Tool: serialize_dataset

#### Arguments

```json
{
  "format": "turtle"
}
```

#### Message

```json
{
  "content": "format='turtle' content='@prefix foaf: <http://xmlns.com/foaf/0.1/> .\\n@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\\n\\n<urn:ex:Animal> a rdfs:Class ;\\n    rdfs:label \"Animal\" ;\\n    rdfs:comment \"A living organism capable of loci, typically multi c\u00e9llulare.\" .\\n\\n<https://example.com/ontology#HomoSapiens> rdfs:subClassOf <https://example.com/ontology#Hominina>,\\n        <https://example.com/ontology#Mammal>,\\n        <urn:ex:Animal> .\\n\\n<urn:ex:john> a foaf:Person ;\\n    rdfs:label \"John\" .\\n\\n<https://example.com/ontology#Hominidae> rdfs:subClassOf <https://example.com/ontology#Haplorhini>,\\n        <https://example.com/ontology#Primate> .\\n\\n<https://example.com/ontology#Hominina> rdfs:subClassOf <https://example.com/ontology#Hominidae> .\\n\\n<https://example.com/ontology#Primate> rdfs:subClassOf <https://example.com/ontology#Mammal> .\\n\\n<https://example.com/ontology#Mammal> rdfs:subClassOf <urn:ex:Animal> .\\n\\n' default_graph_triple_count=13 is_empty=False message='Serialized the current default graph containing 13 triples.'",
  "status": "success",
  "tool_call_id": "H7evBwZQF"
}
```

### Tool: add_triples

#### Message

```json
{
  "content": "requested=3 updated=3 unchanged=0 no_action_needed=False message='3 of 3 triples added.'",
  "status": "success",
  "tool_call_id": "ITWvlVCX3"
}
```

### Tool: inspect_term

#### Message

```json
{
  "content": "{\"uri\":\"<https://example.com/ontology#HomoSapiens>\",\"label\":\"HomoSapiens\",\"definition\":\"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#HomoSapiens</literal_definition_missing>\",\"termType\":\"class\",\"vocabulary\":\"<https://example.com/ontology>\",\"superTerms\":[],\"domain\":[],\"range\":[],\"source_rdf\":{\"format\":\"turtle\",\"content\":\"@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\\n\\n<https://example.com/ontology#HomoSapiens> a rdfs:Class ;\\n    rdfs:isDefinedBy <https://example.com/ontology> .\\n\\n\",\"default_graph_triple_count\":2,\"is_empty\":false,\"message\":\"Serialized the current default graph containing 2 triples.\"}}",
  "status": "success",
  "tool_call_id": "G5UwEOY8E"
}
```

### Tool: list_terms

#### Message

```json
{
  "content": "[{\"definition\": \"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Haplorhini</literal_definition_missing>\", \"label\": \"Haplorhini\", \"termType\": \"class\", \"uri\": \"<https://example.com/ontology#Haplorhini>\"}, {\"definition\": \"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Hominidae</literal_definition_missing>\", \"label\": \"Hominidae\", \"termType\": \"class\", \"uri\": \"<https://example.com/ontology#Hominidae>\"}, {\"definition\": \"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Hominina</literal_definition_missing>\", \"label\": \"Hominina\", \"termType\": \"class\", \"uri\": \"<https://example.com/ontology#Hominina>\"}, {\"definition\": \"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#HomoSapiens</literal_definition_missing>\", \"label\": \"HomoSapiens\", \"termType\": \"class\", \"uri\": \"<https://example.com/ontology#HomoSapiens>\"}, {\"definition\": \"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Mammal</literal_definition_missing>\", \"label\": \"Mammal\", \"termType\": \"class\", \"uri\": \"<https://example.com/ontology#Mammal>\"}, {\"definition\": \"<literal_definition_missing>The vocabulary has no definition for https://example.com/ontology#Primate</literal_definition_missing>\", \"label\": \"Primate\", \"termType\": \"class\", \"uri\": \"<https://example.com/ontology#Primate>\"}]",
  "status": "success",
  "tool_call_id": "niGT0wKwF"
}
```

### Tool: list_vocabularies

#### Message

```json
{
  "content": "{\"vocabularies\":[{\"namespace\":\"<http://www.w3.org/1999/02/22-rdf-syntax-ns#>\",\"label\":\"RDF\",\"description\":\"Core RDF data model terms for statements, lists, containers, and literal/datatype machinery.\",\"term_count\":31},{\"namespace\":\"<http://www.w3.org/2000/01/rdf-schema#>\",\"label\":\"RDFS\",\"description\":\"Schema-level RDF terms for classes, properties, labels, comments, domain/range, and hierarchy modeling.\",\"term_count\":15},{\"namespace\":\"<http://www.w3.org/2002/07/owl#>\",\"label\":\"OWL\",\"description\":\"Ontology modeling and logical constraint terms for classes, restrictions, axioms, and richer property semantics.\",\"term_count\":86},{\"namespace\":\"<http://www.w3.org/ns/prov#>\",\"label\":\"PROV-O\",\"description\":\"Provenance terms for entities, activities, agents, and qualified influence relationships.\",\"term_count\":94},{\"namespace\":\"<http://xmlns.com/foaf/0.1/>\",\"label\":\"FOAF\",\"description\":\"People, agents, profiles, social connections, and related online identity and metadata terms.\",\"term_count\":72},{\"namespace\":\"<https://example.com/ontology>\",\"label\":\"Human Biological Ontology\",\"description\":\"Biological classes for human ontologies\",\"term_count\":6}]}",
  "status": "success",
  "tool_call_id": "zIiULxodt"
}
```


2026-04-18 21:52:20,297 DEBUG rdflib_reasoning.middleware.rdf_vocabulary_middleware: Listing 6 indexed RDF vocabularies
2026-04-18 21:52:22,805 DEBUG rdflib_reasoning.middleware.rdf_vocabulary_middleware: Listing RDF terms for vocabulary https://example.com/ontology (term_type=class, offset=0, limit=20)
2026-04-18 21:52:32,022 DEBUG rdflib_reasoning.middleware.rdf_vocabulary_middleware: Inspecting RDF term https://example.com/ontology#HomoSapiens (include_source_rdf=True)
2026-04-18 21:52:42,625 DEBUG rdflib_reasoning.middleware.dataset_middleware: Adding 3 triples to the default graph
2026-04-18 21:52:59,995 DEBUG rdflib_reasoning.middleware.dataset_middleware: Adding 10 triples to the default graph
2026-04-18 21:52:59,996 DEBUG rdflib_reasoning.middleware.rdf_vocabulary_middleware: Converting vocabulary whitelist violation for tool add_triples and term http://www.w3.org/1000/01/rdf-schema#subClassOf into a tool-facing error
2026-04-18 21:53:13,853 DEBUG rdflib_reasoning.middleware.da

### 2.4. Accessing the Knowledge Base

As in the [dataset middleware demo](./demo-dataset-middleware.ipynb), we retained a reference to the middleware to extract the structured knowledge base without asking the agent to understand RDF serialization formats.

Operationally, that preserves the main gain from the previous two middleware conditions: we still get a machine-usable dataset that can be inspected and scored directly.
What this saved dataset shows, this time, is that the Research Agent did anchor the biological taxonomy in the supplied domain ontology.

The biological classes in the saved graph are the `exterms:` IRIs `Haplorhini`, `Hominidae`, `Hominina`, `HomoSapiens`, `Mammal`, and `Primate`, all drawn from the supplied `https://example.com/ontology` TBox.
The only local taxonomic class the agent mints is `urn:ex:Animal`, which the supplied ontology does not cover.
John is typed as `foaf:Person` from the bundled FOAF vocabulary, which is a reasonable instance-typing choice but does not match the evaluation target's `ex:Person` pattern.
So the remaining gap in this condition is narrower than in the generic-vocabulary notebook: it is about person modeling and OWL-level typings, not about whether the agent reused the supplied ontology at all.

In [12]:
from rdflib import Dataset

dataset: Dataset = dataset_middleware.session.snapshot_dataset()

print("There are", len(dataset), "triples in the dataset")
print("-" * 80)
print(dataset.serialize(format="turtle"))

actual_graph: Graph = dataset.default_graph
actual_graph.bind("exterms", domain_ns)
actual_graph.bind("ex", "urn:ex:")

There are 13 triples in the dataset
--------------------------------------------------------------------------------
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<urn:ex:Animal> a rdfs:Class ;
    rdfs:label "Animal" ;
    rdfs:comment "A living organism capable of loci, typically multi céllulare." .

<https://example.com/ontology#HomoSapiens> rdfs:subClassOf <https://example.com/ontology#Hominina>,
        <https://example.com/ontology#Mammal>,
        <urn:ex:Animal> .

<urn:ex:john> a foaf:Person ;
    rdfs:label "John" .

<https://example.com/ontology#Hominidae> rdfs:subClassOf <https://example.com/ontology#Haplorhini>,
        <https://example.com/ontology#Primate> .

<https://example.com/ontology#Hominina> rdfs:subClassOf <https://example.com/ontology#Hominidae> .

<https://example.com/ontology#Primate> rdfs:subClassOf <https://example.com/ontology#Mammal> .

<https://example.com/ontology#Mammal> rdfs:subClassOf <urn:ex:Ani

### 2.5. Metrics

The evaluation follows the same protocol as the previous conditions: set-based overlap metrics between the extracted graph and the evaluation target, with best-practice documentation triples excluded from penalization.
For this notebook, that target should be read as the earlier target graph together with the adjoined renaming of target classes.

That makes this notebook easy to compare directionally with the previous vocabulary condition even though the intervention is stronger here.
The important question is whether domain-relevant vocabulary access changes the graph in a useful way once machine-usable output has already been secured by dataset-backed state.
In this run, the answer is a small additional precision gain on top of the generic-vocabulary notebook: the graph remains usable and inspectable, intersection is comparable to the generic-vocabulary run, and the extras count drops further as the reused `exterms:` classes absorb what would otherwise have been a minted local class.

In [ ]:
from rdflib.plugins.stores.memory import Memory
from rdflib_reasoning.engine import (
    PRODUCTION_RDFS_RULES,
    RETEEngineFactory,
)
from rdflib_reasoning.engine.rete_store import RETEStore


def deductive_closure(graph: Graph) -> Graph:
    factory = RETEEngineFactory(rules=PRODUCTION_RDFS_RULES)
    store = RETEStore(Memory(), factory)
    dataset = Dataset(store=store)
    output = dataset.default_graph
    for t in graph:
        output.add(t)
    logger.info("Inferred %d triples from the input graph", len(output) - len(graph))
    return output


actual_closure = deductive_closure(actual_graph)
expected_closure = deductive_closure(pseudo_expected)

2026-04-18 21:53:22,589 INFO __main__: Inferred 16 triples from the input graph
2026-04-18 21:53:22,669 INFO __main__: Inferred 30 triples from the input graph


In [14]:
from demo_utils import evaluate_actual_graph

# measures, graphs = evaluate_actual_graph(actual_graph, expected_graph=pseudo_expected)
measures, graphs = evaluate_actual_graph(
    actual_closure, expected_graph=expected_closure
)

intersection = graphs["intersection"]
missing = graphs["missing"]
extra = graphs["extra"]
union = graphs["union"]

#### 2.5.1. Intersection with Target

This run intersects the evaluation target on 25 triples, compared with 27 in the dataset-only notebook and 25 in the generic-vocabulary notebook.
Absolute intersection is therefore comparable to the generic-vocabulary run; the biggest practical gain in this condition is precision rather than overlap volume.

Qualitatively, the overlap comes from direct reuse of the supplied domain TBox.
The saved intersection includes the `exterms:` class declarations for `Haplorhini`, `Hominidae`, `Hominina`, `HomoSapiens`, `Mammal`, and `Primate`, together with their subclass chain back through `ex:Animal`.
What remains missing is not the broad biological structure, but the final layer of alignment around person modeling, OWL-level class typings, and the `owl:equivalentClass` link between `ex:Person` and `exterms:HomoSapiens`.

In [15]:
import sys

if len(intersection) > 0:
    print(f"{len(intersection)} triples in the intersection")
    print("-" * 80)
    print(intersection.serialize(format="turtle"))
else:
    print(
        "The agent's graph output did NOT intersect the expected graph.",
        file=sys.stderr,
    )

25 triples in the intersection
--------------------------------------------------------------------------------
@prefix ex: <urn:ex:> .
@prefix exterms: <https://example.com/ontology#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

exterms:Haplorhini a rdfs:Class .

exterms:Hominidae a rdfs:Class ;
    rdfs:subClassOf exterms:Haplorhini,
        exterms:Mammal,
        exterms:Primate,
        ex:Animal .

exterms:Hominina a rdfs:Class ;
    rdfs:subClassOf exterms:Haplorhini,
        exterms:Hominidae,
        exterms:Mammal,
        exterms:Primate,
        ex:Animal .

exterms:HomoSapiens a rdfs:Class ;
    rdfs:subClassOf exterms:Haplorhini,
        exterms:Hominidae,
        exterms:Hominina,
        exterms:Mammal,
        exterms:Primate,
        ex:Animal .

exterms:Mammal a rdfs:Class ;
    rdfs:subClassOf ex:Animal .

exterms:Primate a rdfs:Class ;
    rdfs:subClassOf exterms:Mammal,
        ex:Animal .

ex:Animal a rdfs:Class .




#### 2.5.2. Missing Facts

The missing-triples count is 26 here, essentially the same as the 26 missing in the generic-vocabulary notebook and 24 in the dataset-only notebook.
So domain-specific vocabulary does not expand the set of target triples the Research Agent recovers; it affects which classes are minted versus reused, not how deeply the ontology is realized.

The remaining misses split into four interpretable groups:

1. `ex:Person`: the target declares it as `rdfs:Class`, attaches the full subclass chain through `exterms:Haplorhini`, `exterms:Hominidae`, `exterms:Hominina`, `exterms:HomoSapiens`, `exterms:Mammal`, `exterms:Primate`, and `ex:Animal`, and asserts `owl:equivalentClass exterms:HomoSapiens`. None of this appears in the saved graph.
2. John's membership chain: the target asserts `ex:John a` each of `exterms:Haplorhini`, `exterms:Hominidae`, `exterms:Hominina`, `exterms:HomoSapiens`, `exterms:Mammal`, `exterms:Primate`, `ex:Animal`, and `ex:Person`. The saved graph only types John as `foaf:Person`.
3. OWL typings on the reused `exterms:` classes: the target declares each of `Haplorhini`, `Hominidae`, `Hominina`, `HomoSapiens`, `Mammal`, and `Primate` as `owl:Class` alongside the `rdfs:Class` assertions that the agent does emit.
4. Extra `rdfs:subClassOf` links that the target attaches to `exterms:Haplorhini` (to `exterms:Mammal`, `exterms:Primate`, and `ex:Animal`).

These are structural gaps in OWL typing and person modeling rather than a failure to select or familiarize with the supplied ontology.

In [16]:
print(f"{len(missing)} missing target triples:")
print("-" * 80)
print(missing.serialize(format="turtle"))

26 missing target triples:
--------------------------------------------------------------------------------
@prefix ex: <urn:ex:> .
@prefix exterms: <https://example.com/ontology#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Person a rdfs:Class ;
    rdfs:subClassOf exterms:Haplorhini,
        exterms:Hominidae,
        exterms:Hominina,
        exterms:HomoSapiens,
        exterms:Mammal,
        exterms:Primate,
        ex:Animal ;
    owl:equivalentClass exterms:HomoSapiens .

ex:John a exterms:Haplorhini,
        exterms:Hominidae,
        exterms:Hominina,
        exterms:HomoSapiens,
        exterms:Mammal,
        exterms:Primate,
        ex:Animal,
        ex:Person .

exterms:Haplorhini a owl:Class ;
    rdfs:subClassOf exterms:Mammal,
        exterms:Primate,
        ex:Animal .

exterms:Hominidae a owl:Class .

exterms:Hominina a owl:Class .

exterms:HomoSapiens a owl:Class .

exterms:Mammal a owl:Class .

ext

#### 2.5.3. Extra Facts

The extra-triples count is 1 in this condition, down from 2 in the generic-vocabulary notebook and 17 in the dataset-only notebook.
The single extra is `ex:john a foaf:Person`.

That one triple is a defensible instance-typing choice: `foaf:Person` is a widely reused class for person instances and is part of the vocabulary set the middleware surfaced.
It is nonetheless an extra relative to the evaluation target, which expects John typed as `ex:Person` and the biological class chain.
The headline observation is that the domain-specific TBox absorbed what was a minted `ex:Primates` extra in the generic-vocabulary run, so the remaining disagreement is about person modeling rather than about minting local taxonomic classes.

In [17]:
print(f"{len(extra)} extra triples:")
print("-" * 80)
print(extra.serialize(format="turtle"))

1 extra triples:
--------------------------------------------------------------------------------
@prefix ex: <urn:ex:> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .

ex:john a foaf:Person .




#### 2.5.4. Summary Metrics

The headline pattern is a small additional precision gain on top of the generic-vocabulary notebook.

1. The run remains operationally successful in the same way as the earlier middleware notebooks, because the graph is machine-usable and directly inspectable.
2. Intersection holds at 25 (comparable to the 25 in the generic-vocabulary notebook and 27 in the dataset-only notebook), while extras drop from 2 to 1 as the reused `exterms:` classes absorb what had been a minted `ex:Primates` extra.
3. The overall score is `F1 = 0.649` here versus `0.641` in the generic-vocabulary notebook and `0.568` in the dataset-only notebook. The improvement is a small precision gain (0.962 versus 0.926) while recall is flat (0.490 versus 0.490).
4. The remaining disagreement is primarily a person-modeling choice (`foaf:Person` versus the target's `ex:Person`) together with the missing `owl:Class` typings and `owl:equivalentClass` assertion, rather than ontology selection.

That supports a narrower claim than "the ontology solved extraction."
A better reading is that domain-relevant vocabulary access makes the reused-term choice cleaner without, on its own, unlocking deeper OWL-level structure or changing the person-modeling decision.

As in the earlier notebooks, an RDFS- or OWL-aware evaluation pass could shift some of these numbers.
But even under the current strict set-based comparison, the direction along this series is consistently improving: `F1 = 0.568`, `0.641`, `0.649`.

In [18]:
from demo_utils import pprint_measures

pprint_measures(measures)

Expected triples: 51
Actual triples: 26
Intersection triples: 25
Missing triples: 26
Extra triples: 1
Union triples: 52
Precision: 0.962
Recall: 0.490
F1 score: 0.649
Jaccard Index: 0.481
Triple Edit Distance: 27
Normalized Triple Edit Distance: 0.519
Exact Match: False


## 3. Conclusions

This notebook shows that richer vocabulary inventory matters.
`RDFVocabularyMiddleware` looked only modestly useful when the indexed vocabularies were generic; once the middleware exposes a small domain-specific biological TBox, the Research Agent familiarizes with it (a `list_terms` pass over the supplied ontology and an `inspect_term` on `exterms:HomoSapiens`) and reuses its classes throughout the saved graph.

That is a meaningful portfolio result for this series.
It shows not just that middleware can make runs inspectable, but that the quality of the agent's tool-visible knowledge surface affects which classes it mints versus reuses.
The metric effect is a small additional precision gain (`F1 = 0.649` here versus `0.641` in the generic-vocabulary notebook), driven almost entirely by one fewer extra triple as the reused `exterms:` classes absorb what had been a minted `ex:Primates` extra.
Recall is unchanged: domain vocabulary did not, on its own, pull in the OWL typings, subclass-chain elaboration, or person-modeling pattern that the evaluation target expects.

At the same time, this notebook should be read carefully.
It is a stronger vocabulary condition rather than a directly comparable fourth benchmark row, because the indexed knowledge available to the agent has changed substantially.
It also relies on an evaluation target that preserves the earlier semantic objective while adjoining a renaming of target classes for this condition.
That caveat does not weaken the qualitative lesson: if the goal is to build Research Agents that ground their outputs in controlled vocabularies, then vocabulary access alone is not enough; the available vocabulary inventory also has to be relevant to the task.